Check the model artifact

In [1]:
import hashlib
import pickle
import subprocess
from pathlib import Path

expected = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual = hashlib.sha256(Path("pipeline.bin").read_bytes()).hexdigest()

actual == expected, actual

(True, '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1')

1. uv version

In [2]:
print(subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())

uv 0.11.32 (x86_64-unknown-linux-gnu)


2. Locked scikit-learn version

In [3]:
import tomllib

with open("uv.lock", "rb") as f:
    lock = tomllib.load(f)

[p["version"] for p in lock["package"] if p["name"] == "scikit-learn"]

['1.7.2']

In [4]:
import sklearn

sklearn.__version__

'1.7.2'

3. Load the model

In [5]:
from model import normalize_record

with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

In [6]:
proba = pipeline.predict_proba([normalize_record(lead)])[0, 1]

proba, round(float(proba), 3)

(np.float64(0.5329944186432507), 0.533)

4. Serve the model

In [7]:
import json
import sys
import time
from urllib.request import Request, urlopen

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        urlopen("http://localhost:9696/health")
        break
    except Exception:
        time.sleep(1)

In [8]:
json.load(urlopen("http://localhost:9696/health"))

{'status': 'ok',
 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1',
 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}

In [9]:
client = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

request = Request(
    "http://localhost:9696/predict",
    data=json.dumps(client).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

result = json.load(urlopen(request))

result, round(result["conversion_probability"], 3)

({'conversion_probability': 0.769799, 'conversion': True}, 0.77)

5. Dockerfile base image

In [10]:
Path("Dockerfile").read_text().splitlines()[0]

'FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3'

6. Run the container

Docker was not available in this environment, so the container was not built. The image uses the same `uv.lock`, `predict.py` and `pipeline.bin` as the local API above, so this is the same request against the same code (`python q6_test.py` does the same thing).

In [11]:
print(subprocess.run([sys.executable, "q6_test.py"], capture_output=True, text=True).stdout.strip())

{'conversion_probability': 0.769799, 'conversion': True}


In [12]:
server.terminate()
server.wait()
print("server stopped")

server stopped
